<a href="https://colab.research.google.com/github/rudra629/ml-internship-flyrank/blob/main/work/notebooks/Untitled9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I am choosing the Refresh / Content Opportunity Scoring lane. I chose this lane because content and SEO teams managing large domains cannot manually audit every URL every month. By framing content triage as a scoring task based on search analytics data, we can build a systematic engine to prioritize human review for pages that are clearly wasting their search visibility.


The Question: Which web pages are wasting their search visibility (high impressions, low CTR) and need a content refresh?

Unit of Analysis: A single web page (content_hash_id) over a 90-day window.

The Decision: Determining exactly which pages the content team should manually review and audit in the current cycle.

The Action: A human writer auditing the page to rewrite meta tags, improve search intent alignment, or update on-page depth.

Cost of a Wrong Call: Wasted human effort. If the model flags a page that is actually fine, a content writer spends 15-30 minutes auditing it unnecessarily. It is a low-risk, high-reward trade-off.

In [1]:
import duckdb
import getpass


hf_token = getpass.getpass("Enter your Hugging Face read token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")


REL = 'hf://datasets/FlyRank/internship-warehouse'
query = f"""
SELECT
    COUNT(DISTINCT content_hash_id) as total_pages,
    AVG(gsc_impressions) as avg_impressions,
    AVG((gsc_clicks*1.0) / NULLIF(gsc_impressions, 0)) as avg_ctr
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE gsc_impressions > 1000
"""
df = con.sql(query).df()

print(f"Total High-Visibility Pages: {df['total_pages'][0]:,.0f}")
print(f"Average Impressions per Page: {df['avg_impressions'][0]:,.0f}")
print(f"Average CTR: {df['avg_ctr'][0]:.2%}")

Enter your Hugging Face read token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total High-Visibility Pages: 8,422
Average Impressions per Page: 1,839
Average CTR: 0.35%


What I can claim: This model provides a directional, decision-support score based on observed search metrics. It reliably identifies patterns where high visibility is met with poor engagement.

What I cannot claim: This is not a causal proof of search engine algorithms, nor does it "predict Google." The model cannot guarantee that updating the content will automatically increase rankings or traffic.

[x] Picked a predefined lane.

[x] Named the decision and the action.

[x] Showed real numbers from the data.

[x] Explained why this is not just "train a model" (it's a triage engine).

[x] Used careful language (observed, directional, decision-support).